In [3]:
# unify_causal_score.py

import json
import numpy as np

ALPHA = 0.05  # significance threshold

with open("Causal_score.json") as f:
    causal_scores = json.load(f)
with open("Causal_pvalues.json") as f:
    causal_pvalues = json.load(f)

unified = {}
for layer in causal_scores:
    scores = np.array(causal_scores[layer])
    pvalues = np.array(causal_pvalues[layer])
    # Where not significant, treat the effect as "no measurable importance" (0)
    # rather than trusting a possibly noisy nonzero number.
    unified[layer] = np.where(pvalues <= ALPHA, scores, 0.0).tolist()

with open("Causal_unified_score.json", "w") as f:
    json.dump(unified, f, indent=4)

In [4]:
# compare_to_gmarpp.py

import json
import numpy as np
from scipy.stats import spearmanr, kendalltau

with open("Causal_unified_score.json") as f:
    causal = json.load(f)
with open("GMARPP_score_tinydeit.json") as f:
    gmarpp = json.load(f)

layers = sorted(causal.keys(), key=int)

# --- Per-layer rank correlation ---
print(f"{'Layer':<6}{'Spearman':<12}{'Kendall':<12}")
for layer in layers:
    c = np.array(causal[layer])
    g = np.array(gmarpp[layer])
    rho, _ = spearmanr(c, g)
    tau, _ = kendalltau(c, g)
    print(f"{layer:<6}{rho:<12.3f}{tau:<12.3f}")

# --- Global rank correlation (flatten all layers, same order) ---
all_causal = np.concatenate([causal[l] for l in layers])
all_gmarpp = np.concatenate([gmarpp[l] for l in layers])
rho, p_rho = spearmanr(all_causal, all_gmarpp)
tau, p_tau = kendalltau(all_causal, all_gmarpp)
print(f"\nGlobal Spearman: {rho:.3f} (p={p_rho:.3g})")
print(f"Global Kendall:  {tau:.3f} (p={p_tau:.3g})")

# --- Agreement on WHICH heads get pruned (more actionable than raw correlation) ---
def bottom_k_heads(score_dict, layers, k_percent=10):
    flat = [(layer, idx, val) for layer in layers for idx, val in enumerate(score_dict[layer])]
    flat_sorted = sorted(flat, key=lambda x: x[2])  # ascending -> least important first
    cutoff = int(len(flat_sorted) * (k_percent / 100))
    return set((layer, idx) for layer, idx, _ in flat_sorted[:cutoff])

for k in [10, 20, 30]:
    causal_bottom = bottom_k_heads(causal, layers, k)
    gmarpp_bottom = bottom_k_heads(gmarpp, layers, k)
    overlap = len(causal_bottom & gmarpp_bottom)
    jaccard = overlap / len(causal_bottom | gmarpp_bottom)
    print(f"Bottom {k}%: {overlap}/{len(causal_bottom)} heads agree "
          f"(Jaccard={jaccard:.3f})")

Layer Spearman    Kendall     
0     -1.000      -1.000      
1     -0.500      -0.333      
2     0.500       0.333       
3     -1.000      -1.000      
4     -0.500      -0.333      
5     -1.000      -1.000      
6     -0.500      -0.333      
7     0.500       0.333       
8     -1.000      -1.000      
9     -0.500      -0.333      
10    -0.500      -0.333      
11    -1.000      -1.000      

Global Spearman: -0.439 (p=0.00735)
Global Kendall:  -0.287 (p=0.0141)
Bottom 10%: 0/3 heads agree (Jaccard=0.000)
Bottom 20%: 0/7 heads agree (Jaccard=0.000)
Bottom 30%: 0/10 heads agree (Jaccard=0.000)
